# 상위20% 위축 예측 — 피처 엔지니어링

**대상**: 초반(week 17~26) 기준 상위20%였던 가구 (약 499명)
**라벨**: 후반(week 90~99) 기준으로도 상위20%를 유지했는가(0=유지) vs 못했는가(1=위축/이탈) — 정의(b)

**사용 feature (EDA로 확정된 것만)**
- transaction_data 파생: 초반 지출액, 방문빈도, 카테고리 다양성, basket당 평균 품목수
- causal_data: 초반 구간 구매 중 매대/전단 노출 상품 비율
- hh_demographic: 인구통계 원본 값 (결측은 "정보없음" 카테고리로 포함)

**제외된 것 (EDA에서 확인된 이유)**
- coupon_redempt, campaign_table: 초반 구간(DAY 111~180)에 활동 자체가 없어 전부 상수(0)이므로 정보량 없음

**모든 feature는 초반 구간(week 17~26)만 사용 — 데이터 누수 방지**


In [11]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 150)

DATA_DIR = "data/"

STABLE_MIN_WEEK = 17
STABLE_MAX_WEEK = 99
EARLY_N = 10
TOP_X_PCT = 20

tx = pd.read_csv(DATA_DIR + "transaction_data.csv")
hh_demo = pd.read_csv(DATA_DIR + "hh_demographic.csv")
causal = pd.read_csv(DATA_DIR + "causal_data.csv")
product = pd.read_csv(DATA_DIR + "product.csv")

# 컬럼명 대소문자 통일 (causal_data가 소문자였던 이슈 방지)
causal.columns = [c.upper() for c in causal.columns]
tx.columns = [c.upper() for c in tx.columns]
hh_demo.columns = [c.upper() for c in hh_demo.columns]
product.columns = [c.upper() for c in product.columns]

tx_stable = tx[(tx["WEEK_NO"] >= STABLE_MIN_WEEK) & (tx["WEEK_NO"] <= STABLE_MAX_WEEK)].copy()
stable_households = tx_stable["HOUSEHOLD_KEY"].unique()
print(f"안정구간 가구 수: {len(stable_households):,}")


안정구간 가구 수: 2,492


## 1. 초반 상위20% 가구(라벨링 대상) 재확정

이전 노트북들과 동일한 방식(초반 지출액 기준 상위20%)으로 대상 가구를 다시 뽑는다.


In [12]:
early_window = tx_stable[
    (tx_stable["WEEK_NO"] >= STABLE_MIN_WEEK) &
    (tx_stable["WEEK_NO"] < STABLE_MIN_WEEK + EARLY_N)
].copy()

sales_early_full = early_window.groupby("HOUSEHOLD_KEY")["SALES_VALUE"].sum()

panel_wide = pd.DataFrame(index=stable_households)
panel_wide["sales_early"] = sales_early_full.reindex(panel_wide.index).fillna(0)

cutoff_early = panel_wide["sales_early"].quantile(1 - TOP_X_PCT/100)
panel_wide["group_early"] = np.where(panel_wide["sales_early"] >= cutoff_early, "상위20%", "나머지80%")

top20_households = panel_wide[panel_wide["group_early"] == "상위20%"].index
print(f"초반 기준 상위20% 가구 수: {len(top20_households):,}")
print(f"컷오프 지출액: {cutoff_early:.2f}")


초반 기준 상위20% 가구 수: 499
컷오프 지출액: 535.29


## 2. 라벨 생성 — 후반 기준으로도 상위20%를 유지했는가

후반(week 90~99) 지출액으로 전체 인구 기준 상위20% 컷오프를 다시 계산하고,
원래 상위20%였던 가구들이 후반에도 그 안에 남아있는지 확인.


In [13]:
late_window = tx_stable[
    (tx_stable["WEEK_NO"] > STABLE_MAX_WEEK - EARLY_N) &
    (tx_stable["WEEK_NO"] <= STABLE_MAX_WEEK)
].copy()

sales_late_full = late_window.groupby("HOUSEHOLD_KEY")["SALES_VALUE"].sum()
panel_wide["sales_late"] = sales_late_full.reindex(panel_wide.index).fillna(0)

cutoff_late = panel_wide["sales_late"].quantile(1 - TOP_X_PCT/100)
panel_wide["group_late"] = np.where(panel_wide["sales_late"] >= cutoff_late, "상위20%", "나머지80%")

print(f"후반 기준 상위20% 컷오프 지출액: {cutoff_late:.2f}")

# 라벨: 원래 상위20%였던 가구 중, 후반에도 상위20%면 0(유지), 아니면 1(위축/이탈)
labels = panel_wide.loc[top20_households, "group_late"].apply(
    lambda x: 0 if x == "상위20%" else 1
)
labels.name = "label_shrink"

print("\n라벨 분포 (0=유지, 1=위축/이탈)")
print(labels.value_counts())
print(f"위축 비율: {labels.mean()*100:.1f}%")


후반 기준 상위20% 컷오프 지출액: 627.88

라벨 분포 (0=유지, 1=위축/이탈)
label_shrink
0    276
1    223
Name: count, dtype: int64
위축 비율: 44.7%


## 3. Feature 1~2 — 방문빈도, basket당 평균 품목수 (transaction_data)

In [14]:
early_top20 = early_window[early_window["HOUSEHOLD_KEY"].isin(top20_households)].copy()

# QUANTITY 합산 기반 avg_qty_per_basket은 무게/부피 단위 상품(정육, 벌크 등) 때문에
# 극단치가 심하게 섞여 있어(예: QUANTITY=41,686) 사용하지 않음.
# 대신 "basket당 평균 고유 상품 개수(품목 다양성)"로 대체 — 단위 문제에서 자유로움.

feat_visit = early_top20.groupby("HOUSEHOLD_KEY").agg(
    n_baskets=("BASKET_ID", "nunique"),
).reset_index().set_index("HOUSEHOLD_KEY")

items_per_basket = (
    early_top20.groupby(["HOUSEHOLD_KEY", "BASKET_ID"])["PRODUCT_ID"]
    .nunique()
    .reset_index()
)
avg_items = items_per_basket.groupby("HOUSEHOLD_KEY")["PRODUCT_ID"].mean().rename("avg_items_per_basket")

feat_visit = feat_visit.join(avg_items)

print(feat_visit.describe())
feat_visit.head()


        n_baskets  avg_items_per_basket
count  499.000000            499.000000
mean    29.332665             13.507945
std     23.109256              8.263343
min      4.000000              1.208333
25%     16.000000              8.057143
50%     23.000000             11.238095
75%     35.000000             16.572917
max    184.000000             48.750000


,n_baskets,avg_items_per_basket
HOUSEHOLD_KEY,,
6,30,9.500000
13,36,9.083333
17,15,6.133333
18,15,13.333333
19,25,12.880000


## 4. Feature 3 — 카테고리 다양성 (product 조인)

In [15]:
early_top20_prod = early_top20.merge(
    product[["PRODUCT_ID", "COMMODITY_DESC"]], on="PRODUCT_ID", how="left"
)

feat_diversity = early_top20_prod.groupby("HOUSEHOLD_KEY")["COMMODITY_DESC"].nunique()
feat_diversity.name = "category_diversity"

print(feat_diversity.describe())


count    499.000000
mean      83.715431
std       20.618235
min        5.000000
25%       70.000000
50%       82.000000
75%       96.000000
max      158.000000
Name: category_diversity, dtype: float64


## 5. Feature 4 — 매대/전단 노출 상품 구매 비율 (causal_data 조인)

초반 구간(week 17~26)의 causal_data만 필터링해서, 각 거래 라인이 노출(display 또는 mailer ≠ '0')된
상품·매장·주 조합이었는지 플래그를 붙이고, 가구별 평균(노출 비율)을 계산.


In [16]:
causal_early = causal[
    (causal["WEEK_NO"] >= STABLE_MIN_WEEK) &
    (causal["WEEK_NO"] < STABLE_MIN_WEEK + EARLY_N)
][["PRODUCT_ID", "STORE_ID", "WEEK_NO", "DISPLAY", "MAILER"]].copy()

# 코드값을 문자열로 통일 (0 vs '0' 비교 오류 방지)
causal_early["DISPLAY"] = causal_early["DISPLAY"].astype(str)
causal_early["MAILER"] = causal_early["MAILER"].astype(str)

early_top20_causal = early_top20.merge(
    causal_early, on=["PRODUCT_ID", "STORE_ID", "WEEK_NO"], how="left"
)

# 매칭 안 된(causal_data에 없는) 거래는 노출 정보 없음 → 미노출로 간주
early_top20_causal["DISPLAY"] = early_top20_causal["DISPLAY"].fillna("0")
early_top20_causal["MAILER"] = early_top20_causal["MAILER"].fillna("0")

early_top20_causal["exposed"] = (
    (early_top20_causal["DISPLAY"] != "0") | (early_top20_causal["MAILER"] != "0")
).astype(int)

feat_exposure = early_top20_causal.groupby("HOUSEHOLD_KEY")["exposed"].mean()
feat_exposure.name = "exposure_ratio"

print(feat_exposure.describe())


count    499.000000
mean       0.217594
std        0.078652
min        0.000000
25%        0.162174
50%        0.212121
75%        0.266591
max        0.462963
Name: exposure_ratio, dtype: float64


## 6. Feature 5~10 — 인구통계 (hh_demographic, 결측은 '정보없음')

In [17]:
demo_cols = ["AGE_DESC", "INCOME_DESC", "HOMEOWNER_DESC", "HH_COMP_DESC",
             "HOUSEHOLD_SIZE_DESC", "KID_CATEGORY_DESC"]

feat_demo = hh_demo.set_index("HOUSEHOLD_KEY")[demo_cols].reindex(top20_households)
feat_demo = feat_demo.fillna("정보없음")

print("결측(정보없음) 처리 후 각 컬럼 분포 확인")
for c in demo_cols:
    print(f"\n--- {c} ---")
    print(feat_demo[c].value_counts())


결측(정보없음) 처리 후 각 컬럼 분포 확인

--- AGE_DESC ---
AGE_DESC
정보없음     180
45-54    118
35-44     91
25-34     58
55-64     20
65+       19
19-24     13
Name: count, dtype: int64

--- INCOME_DESC ---
INCOME_DESC
정보없음         180
50-74K        84
35-49K        54
75-99K        36
25-34K        25
Under 15K     24
15-24K        23
125-149K      21
100-124K      19
150-174K      18
250K+          8
175-199K       5
200-249K       2
Name: count, dtype: int64

--- HOMEOWNER_DESC ---
HOMEOWNER_DESC
Homeowner          216
정보없음               180
Unknown             79
Renter              19
Probable Owner       3
Probable Renter      2
Name: count, dtype: int64

--- HH_COMP_DESC ---
HH_COMP_DESC
정보없음                180
2 Adults No Kids    108
2 Adults Kids        82
Single Female        47
Single Male          35
Unknown              26
1 Adult Kids         21
Name: count, dtype: int64

--- HOUSEHOLD_SIZE_DESC ---
HOUSEHOLD_SIZE_DESC
정보없음    180
2       132
1        89
3        47
5+       28
4        2

## 7. 전체 feature 테이블 합치기

In [18]:
feature_table = panel_wide.loc[top20_households, ["sales_early"]].copy()
feature_table = feature_table.join(feat_visit)
feature_table = feature_table.join(feat_diversity)
feature_table = feature_table.join(feat_exposure)
feature_table = feature_table.join(feat_demo)
feature_table = feature_table.join(labels)

print(f"최종 feature table shape: {feature_table.shape}")
print("\n결측 확인 (0이어야 정상 — 인구통계는 이미 '정보없음'으로 채움)")
print(feature_table.isna().sum())

feature_table.head(10)


최종 feature table shape: (499, 12)

결측 확인 (0이어야 정상 — 인구통계는 이미 '정보없음'으로 채움)
sales_early             0
n_baskets               0
avg_items_per_basket    0
category_diversity      0
exposure_ratio          0
AGE_DESC                0
INCOME_DESC             0
HOMEOWNER_DESC          0
HH_COMP_DESC            0
HOUSEHOLD_SIZE_DESC     0
KID_CATEGORY_DESC       0
label_shrink            0
dtype: int64


,sales_early,n_baskets,avg_items_per_basket,category_diversity,exposure_ratio,AGE_DESC,INCOME_DESC,HOMEOWNER_DESC,HH_COMP_DESC,HOUSEHOLD_SIZE_DESC,KID_CATEGORY_DESC,label_shrink
232,2426.34,159,5.672956,144,0.226164,35-44,35-49K,Unknown,2 Adults No Kids,2,None/Unknown,1
2318,2596.66,91,9.021978,120,0.237515,19-24,Under 15K,Unknown,Single Female,1,None/Unknown,1
697,686.07,9,27.444444,79,0.105263,정보없음,정보없음,정보없음,정보없음,정보없음,정보없음,1
1804,1335.60,46,6.760870,106,0.212219,65+,35-49K,Homeowner,2 Adults No Kids,2,None/Unknown,0
510,1149.80,35,13.457143,114,0.178344,35-44,250K+,Homeowner,2 Adults Kids,3,1,0
762,625.26,69,2.289855,56,0.183544,35-44,125-149K,Renter,Single Female,2,None/Unknown,0
1453,1931.06,26,22.307692,136,0.181034,45-54,125-149K,Homeowner,2 Adults Kids,3,1,0
72,740.09,20,13.700000,79,0.266423,정보없음,정보없음,정보없음,정보없음,정보없음,정보없음,1
603,1050.97,42,9.023810,88,0.208443,정보없음,정보없음,정보없음,정보없음,정보없음,정보없음,1
694,619.60,51,4.725490,63,0.215768,정보없음,정보없음,정보없음,정보없음,정보없음,정보없음,1


## 8. 라벨별 feature 기초 통계 (인코딩 전 탐색)

본격 인코딩·모델링 전에, 라벨(유지 vs 위축)에 따라 수치형 feature들이 눈에 띄게 다른지 미리 확인.


In [19]:
numeric_cols = ["sales_early", "n_baskets", "avg_items_per_basket", "category_diversity", "exposure_ratio"]

print(feature_table.groupby("label_shrink")[numeric_cols].agg(["mean", "median"]))


              sales_early          n_baskets        avg_items_per_basket            category_diversity        exposure_ratio          
                     mean  median       mean median                 mean     median               mean median           mean    median
label_shrink                                                                                                                          
0             1046.155399  918.32  29.369565   24.0            14.173353  11.746429          88.246377   87.0       0.213161  0.205941
1              825.776502  713.05  29.286996   22.0            12.684391  10.230769          78.107623   77.0       0.223079  0.222222


## 9. 저장 (다음 단계: 인코딩 → VIF → 모델링에서 재사용)

In [20]:
feature_table.index.name = "HOUSEHOLD_KEY"
feature_table.to_csv("h4_shrink_feature_table_raw.csv")
print("저장 완료: h4_shrink_feature_table_raw.csv")
print(f"shape: {feature_table.shape}")


저장 완료: h4_shrink_feature_table_raw.csv
shape: (499, 12)
